# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by @id and name/description
record_sets = list(dataset.record_sets)
print(f"Record Sets ({len(record_sets)} found):")
for record_set in record_sets:
    print(f"  @id: {record_set['@id']}  | name: {record_set.get('name', '<none>')}  | description: {record_set.get('description', '')}")
    if 'field' in record_set:
        print("    Fields:")
        for field in record_set['field']:
            if isinstance(field, dict) and '@id' in field:
                print(f"      - {field['@id']}")
            elif isinstance(field, str):
                print(f"      - {field}")

### Sample Records from the First Record Set
Query and print a few sample records for a first look at the data structure.

In [ ]:
# For demonstration, let's show some records of the 1st available record set:
if record_sets:
    record_set_id = record_sets[0]['@id']  # referencing by @id
    print(f"Sample records from record set {record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i >= 2:  # Only print 3 records
            break
else:
    print("No record sets found in dataset.")

## 3. Data Extraction
Load data from the available record set(s) into a DataFrame for analysis. We continue using entity `@id`s for all references.

In [ ]:
# Extract data from all record sets into pandas DataFrames
dataframes = {}
for record_set in record_sets:
    rset_id = record_set['@id']
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Loaded {len(df)} records from record set @id='{rset_id}'")

# Show columns and preview data for the first record set
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"Fields available in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. You may want to remove outliers, transform distributions, or group by categorical attributes before further analysis.

**All fields and columns are referenced using their `@id`.**

In [ ]:
# Example: Filter, normalize, and group numeric data
import numpy as np

# Use the first record set as the main source of analysis
df = dataframes[main_record_set_id].copy()

# Identify numeric columns by type or by inspecting the DataFrame
print("Columns (inferred numeric shown below):")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(numeric_cols)

if numeric_cols:
    numeric_field = numeric_cols[0]  # Choose the first numeric field for demonstration
    # Outlier threshold (e.g., 3 std above mean)
    threshold = df[numeric_field].mean() + 3 * df[numeric_field].std()
    filtered_df = df[df[numeric_field] < threshold]
    print(f"Filtered records: {numeric_field} < {threshold:.2f} (after outlier removal)")
    print(filtered_df[[numeric_field]].head())

    # Normalize selected numeric field (z-score)
    filtered_df[f'{numeric_field}_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score):")
    print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

    # Attempt to group by a categorical/text column
    possible_groups = df.select_dtypes(include=['object']).columns.tolist()
    group_field = possible_groups[0] if possible_groups else None
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric columns detected in DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Simple visualizations for numeric fields
if numeric_cols:
    plt.figure(figsize=(8,4))
    filtered_df[numeric_field].hist(bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        # Bar plot of mean value grouped by the group field
        grouped_df.plot(x=group_field, y=numeric_field, kind='bar', legend=False, figsize=(10,4))
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.ylabel(f'Mean {numeric_field}')
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric fields to plot.")

## 6. Conclusion
This notebook demonstrated how to load, explore, and process the mass spectrometry dataset using the `mlcroissant` library. Key steps included referencing all dataset entities by their `@id` and performing preliminary EDA and visualization using pandas/matplotlib. For more detailed analyses and field mapping (including advanced record set/field selections), refer to the Croissant schema documentation.